In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import pandas as pd
import numpy as np
import gc
from typing import List, Dict, Any, Optional
from idtxl.multivariate_te import MultivariateTE
from idtxl.data import Data
from idtxl.results import ResultsNetworkInference
from scipy.stats import pearsonr
from scipy.signal import decimate
from statsmodels.tsa.stattools import grangercausalitytests
import h5py
import logging
import sys
from statsmodels.stats.multitest import multipletests
import argparse

# ==============================================================================
# Configuration & Paths
# ==============================================================================
BASE_PATH = r"/root/meg_data"
SAVE_BASE_PATH = r"/root/meg_results"

# [KEY CONFIG] IDTxl Parameters for Multivariate Transfer Entropy (mTE)
SETTINGS = {
    'cmi_estimator': 'OpenCLKraskovCMI', # GPU-accelerated estimator 
    'max_lag_sources': 10,               # Maximum past history to consider for source nodes
    'min_lag_sources': 1,
    'max_lag_target': 10,                # Maximum past history for the target node itself
    'alpha': 0.05,                       # Statistical significance threshold
    'n_perm_max_stat': 200,              # Permutations for statistical testing
    'n_perm_min_stat': 200,
    'n_perm_omnibus': 200,
    'n_perm_max_seq': 200,
    'verbose': True
}

def group_tasks_by_subject(all_task_files: List[str]) -> Dict[str, List[str]]:
    """
    Group task files by subject ID.
    Expected MEG format: 100307_MEG_3-Restin_rmegpreproc_Alpha.parquet
    """
    subject_task_map = {}
    for file in all_task_files:
        subject_id = file.split('_')[0]
        if subject_id not in subject_task_map:
            subject_task_map[subject_id] = []
        subject_task_map[subject_id].append(file)
    return subject_task_map

def apply_fdr_correction(p_matrix: np.ndarray, method: str = 'fdr_bh') -> np.ndarray:
    """Apply Benjamini/Hochberg FDR correction to the p-value matrix to control false positives."""
    flat_pvals = p_matrix.flatten()
    valid_mask = ~np.isnan(flat_pvals)
    valid_pvals = flat_pvals[valid_mask]

    _, corrected_pvals, _, _ = multipletests(valid_pvals, alpha=0.05, method=method)

    corrected_matrix = np.full_like(p_matrix, np.nan)
    corrected_matrix.flat[valid_mask] = corrected_pvals
    return corrected_matrix

def pearson_analysis(data_df: pd.DataFrame, brain_regions: List[str]) -> (np.ndarray, np.ndarray, np.ndarray):
    """Calculate the whole-brain Pearson correlation coefficient and p-value."""
    n_regions = len(brain_regions)
    corr_matrix = np.zeros((n_regions, n_regions))
    pvalue_matrix = np.zeros((n_regions, n_regions))

    for i, region_i in enumerate(brain_regions):
        for j, region_j in enumerate(brain_regions):
            if i <= j:
                corr, p_value = pearsonr(data_df[region_i], data_df[region_j])
                corr_matrix[i, j] = corr
                corr_matrix[j, i] = corr
                pvalue_matrix[i, j] = p_value
                pvalue_matrix[j, i] = p_value

    pearson_fdr_pval = apply_fdr_correction(pvalue_matrix)
    return corr_matrix, pvalue_matrix, pearson_fdr_pval

def granger_analysis(data_df: pd.DataFrame, brain_regions: List[str], max_lag: int = 3) -> (np.ndarray, np.ndarray):
    """Calculate the whole-brain Granger causality and return the p-value matrix."""
    n_regions = len(brain_regions)
    pvalue_matrix = np.full((n_regions, n_regions), np.nan)

    for source_idx, source in enumerate(brain_regions):
        for target_idx, target in enumerate(brain_regions):
            if source_idx == target_idx:
                continue

            ts_pair = data_df[[target, source]].values
            try:
                test_result = grangercausalitytests(ts_pair, maxlag=max_lag, verbose=False)
                p_value = test_result[max_lag][0]['ssr_ftest'][1]
                pvalue_matrix[source_idx, target_idx] = p_value
            except Exception as e:
                pass  # Suppress excessive logs for GCA failures to keep terminal clean

    gca_fdr_pval = apply_fdr_correction(pvalue_matrix)
    return pvalue_matrix, gca_fdr_pval

def select_mte_sources(
        pearson_corr: np.ndarray,
        pearson_fdr_pval: np.ndarray,
        gca_fdr_pval: np.ndarray,
        brain_regions: List[str],
        pearson_threshold: float = 0.2,
        gca_threshold: float = 0.05
) -> Dict[int, List[int]]:
    """
    [HEURISTIC FILTER] 
    mTE is extremely computationally expensive. This function acts as a pre-filter, 
    selecting only the top 10 most likely source candidates for each target region 
    based on prior Pearson and GCA significance.
    """
    sources_dict = {}

    for i, target in enumerate(brain_regions):
        candidates = []
        for j, source in enumerate(brain_regions):
            if i == j:
                continue

            pearson_positive = (abs(pearson_corr[j, i]) >= pearson_threshold and pearson_fdr_pval[j, i] <= 0.05)
            gca_positive = (not np.isnan(gca_fdr_pval[j, i]) and gca_fdr_pval[j, i] <= gca_threshold)

            if pearson_positive or gca_positive:
                candidates.append(j)
                old_len = len(candidates)
                # Sort and cap at top 10 candidates to prevent IDTxl from hanging indefinitely
                if old_len > 10:
                    candidates.sort(
                        key=lambda x: abs(pearson_corr[x, i]) + (
                            -np.log10(max(gca_fdr_pval[x, i], 1e-10)) if not np.isnan(gca_fdr_pval[x, i]) else 0),
                        reverse=True
                    )
                    candidates = candidates[:10]
        sources_dict[i] = candidates
    return sources_dict

def mte_analysis(
        final_ts: np.ndarray,
        brain_regions: List[str],
        sources_dict: Dict[int, List[int]],
        brain_info: Dict[str, str],
        settings: Dict = SETTINGS,
        mte_file_path: str = None
) -> Optional[ResultsNetworkInference]:
    """Execute IDTxl Multivariate Transfer Entropy with incremental HDF5 saving."""
    try:
        # Data format must match IDTxl requirements: (processes, samples, replications)
        data = Data(final_ts, dim_order='psr')
        logging.info(f"[mTE] Start analyzing {len(sources_dict)} targets")

        mte_model = MultivariateTE()
        final_results = None

        for target_idx, source_idxs in sources_dict.items():
            if not source_idxs:
                continue
            
            # Check if this specific target has already been calculated
            if mte_file_path and os.path.exists(mte_file_path):
                try:
                    with h5py.File(mte_file_path, 'a') as f:
                        if f'results/mte_results/target_{target_idx}' in f:
                            logging.info(f"[RESUME] Target {target_idx} exists, skipping")
                            continue
                except OSError:
                    pass
            try:
                target_result = mte_model.analyse_single_target(settings=settings, data=data, target=target_idx, sources=source_idxs)
                
                if final_results is None:
                    final_results = target_result
                else:
                    final_results.combine_results(target_result)
                
                # Save to HDF5 immediately after calculating each target. 
                # This prevents total data loss if the server crashes mid-analysis.
                if mte_file_path:
                    with h5py.File(mte_file_path, 'a') as f:
                        result_group = f.require_group('results')
                        mte_group = result_group.require_group("mte_results")
                        target_group = mte_group.require_group(f"target_{target_idx}")
                        single_res = target_result.get_single_target(target_idx, fdr=False)
                        
                        if "selected_vars_sources" in single_res and single_res["selected_vars_sources"]:
                            sources = np.array(single_res["selected_vars_sources"], dtype=int)
                            target_group.require_dataset("selected_sources_idx_lag", shape=sources.shape, dtype=int, data=sources)
                        
                        if "selected_sources_te" in single_res and single_res["selected_sources_te"] is not None and len(single_res["selected_sources_te"]) > 0:
                            te_vals = np.asarray(single_res["selected_sources_te"], dtype=np.float64)
                            target_group.require_dataset("te_values", shape=te_vals.shape, dtype=np.float64, data=te_vals)
                        
                        if "selected_sources_pval" in single_res and single_res["selected_sources_pval"] is not None and len(single_res["selected_sources_pval"]) > 0:
                            p_vals = np.asarray(single_res["selected_sources_pval"], dtype=np.float64)
                            target_group.require_dataset("p_values", shape=p_vals.shape, dtype=np.float64, data=p_vals)
                        
                        for key in ["omnibus_te", "omnibus_pval", "omnibus_sign"]:
                            if key in single_res:
                                val = single_res[key]
                                if isinstance(val, (np.ndarray, np.generic)):
                                    val = val.item() if val.ndim == 0 else val.tolist()
                                elif isinstance(val, np.number):
                                    val = val.item()
                                try:
                                    target_group.attrs[key] = val
                                except TypeError:
                                    target_group.attrs[key] = str(val)
                        f.flush()

                logging.info(f"[mTE Progress] Target {target_idx} successfully calculated.")

            except Exception as e:
                logging.warning(f"[mTE] Target {target_idx} failed: {e}")

        if final_results is None:
            raise ValueError("No valid mTE results after single-target analysis")

        final_results.brain_regions = brain_regions
        final_results.brain_info = brain_info
        return final_results

    except Exception as e:
        logging.error(f"[mTE] Unexpected error: {e}")
        return None

def save_results(
        save_dir: str,
        filename_prefix: str,
        result_data: Optional[Dict[str, Any]],
        brain_regions: List[str],
        brain_info: Dict[str, str],
        analysis_type: str,
        settings: Dict = SETTINGS
) -> None:
    """Save the finalized analysis results to HDF5."""
    if result_data is None:
        return

    os.makedirs(save_dir, exist_ok=True)
    h5_path = os.path.join(save_dir, f"{filename_prefix}.h5")

    try:
        with h5py.File(h5_path, 'a') as f:
            if 'brain_regions' not in f:
                f.create_dataset('brain_regions', data=np.array(brain_regions, dtype='S'))
            if 'brain_network_map' not in f:
                network_names = [brain_info[r] for r in brain_regions]
                f.create_dataset('brain_network_map', data=np.array(network_names, dtype='S'))

            param_group = f.require_group('parameters')
            for k, v in settings.items():
                param_group.attrs[k] = str(v)
            param_group.attrs['analysis_type'] = analysis_type

            result_group = f.require_group('results')

            if analysis_type in ["Pearson", "GCA"]:
                for key, value in result_data.items():
                    if isinstance(value, np.ndarray):
                        dataset_path = f"results/{key}"
                        if dataset_path in result_group:
                            del result_group[dataset_path]
                        result_group.create_dataset(dataset_path, data=value)

            elif analysis_type in ["mTE"]:
                mte_group = result_group.require_group("mte_results")
                for target in result_data.targets_analysed:
                    target_group = mte_group.require_group(f"target_{target}")
                    single_res = result_data.get_single_target(target, fdr=False)

                    if "selected_sources_idx_lag" not in target_group:
                        if "selected_vars_sources" in single_res and single_res["selected_vars_sources"]:
                            sources = np.array(single_res["selected_vars_sources"], dtype=int)
                            target_group.create_dataset("selected_sources_idx_lag", data=sources)
                        
                        if "selected_sources_te" in single_res and single_res["selected_sources_te"] is not None and len(single_res["selected_sources_te"]) > 0:
                            target_group.create_dataset("te_values", data=np.asarray(single_res["selected_sources_te"], dtype=np.float64))
                        
                        if "selected_sources_pval" in single_res and single_res["selected_sources_pval"] is not None and len(single_res["selected_sources_pval"]) > 0:
                            p_values = np.asarray(single_res["selected_sources_pval"], dtype=np.float64)
                            target_group.create_dataset("p_values", data=p_values)
                            
                        for key in ["omnibus_te", "omnibus_pval", "omnibus_sign"]:
                            if key in single_res:
                                value = single_res[key]
                                if isinstance(value, (np.ndarray, np.generic)):
                                    value = value.item() if value.ndim == 0 else value.tolist()
                                elif isinstance(value, np.number):
                                    value = value.item()
                                try:
                                    target_group.attrs[key] = value
                                except TypeError:
                                    target_group.attrs[key] = str(value)

                # Adjacency Matrices Extraction
                adj = result_data.get_adjacency_matrix(weights='binary', fdr=False)
                if "binary_adjacency_matrix" in result_group:
                    del result_group["binary_adjacency_matrix"]
                result_group.create_dataset("binary_adjacency_matrix", data=adj.edge_matrix.astype(np.uint8),
                                            compression="gzip")

                n_nodes = len(result_data.brain_regions)
                te_weighted_matrix = np.zeros((n_nodes, n_nodes))
                for target_idx in result_data.targets_analysed:
                    single_res = result_data.get_single_target(target_idx, fdr=False)
                    if "selected_vars_sources" in single_res and single_res["selected_vars_sources"]:
                        sources = [src[0] for src in single_res["selected_vars_sources"]]
                        te_values = single_res["selected_sources_te"]
                        for src, te in zip(sources, te_values):
                            te_weighted_matrix[src, target_idx] = te
                
                if "te_weighted_adjacency_matrix" in result_group:
                    del result_group["te_weighted_adjacency_matrix"]
                result_group.create_dataset("te_weighted_adjacency_matrix", data=te_weighted_matrix, compression="gzip")

                result_group.attrs["n_significant_links"] = int(adj.n_edges())
                result_group.attrs['fdr_corrected'] = False
                result_group.attrs['note'] = 'Combined using combine_results without network-level FDR correction.'

                settings_group = result_group.require_group("settings")
                for k, v in result_data.settings.items():
                    try:
                        settings_group.attrs[k] = v
                    except TypeError:
                        settings_group.attrs[k] = str(v)

        logging.info(f"[SAVED] {h5_path}")
    except Exception as e:
        logging.error(f"[SAVE FAIL] {h5_path} | Error: {e}")

def save_mte_selection_summary(summary_dict: dict, save_path: str = None):
    """Save the mTE candidate sources summary."""
    if save_path is None:
        save_path = os.path.join(SAVE_BASE_PATH, "mTE_selection_summary.h5")
    save_dir = os.path.dirname(save_path)
    os.makedirs(save_dir, exist_ok=True)

    with h5py.File(save_path, 'w') as f:
        for subj_id, task_dict in summary_dict.items():
            subj_group = f.create_group(subj_id)
            for task_name, sources in task_dict.items():
                data = sources if sources else ['NONE']
                subj_group.create_dataset(task_name, data=[str(s).encode('utf-8') for s in data])
    logging.info(f"[mTE Summary] Saved: {save_path}")

def parse_subject_selection(subj_input: str, all_subj_ids: List[str]) -> List[str]:
    """Parse command line subject input."""
    selected = []
    if not subj_input or subj_input.lower() == 'all':
        return all_subj_ids

    for part in subj_input.split(','):
        part = part.strip()
        if part in all_subj_ids:
            selected.append(part)
    return list(set(selected))

def run_one_task(task_tuple):
    subj_id, task_name, task_file = task_tuple
    file_path = os.path.join(BASE_PATH, task_file)

    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format=f'%(asctime)s | %(levelname)-8s | PID{{os.getpid():5d}} | SUBJ {subj_id} | TASK {task_name} | %(message)s',
        handlers=[
            logging.FileHandler(f"{SAVE_BASE_PATH}/log_{subj_id}_{task_name}.log", encoding='utf-8'),
            logging.StreamHandler(sys.stdout)
        ],
        force=True
    )

    logging.info(f"START -> Subject {subj_id} | Task {task_name}")
    local_summary = {}

    mte_file = os.path.join(SAVE_BASE_PATH, f"mTE_{subj_id}_{task_name}.h5")
    if os.path.exists(mte_file):
        logging.info(f"[SKIP] mTE already exists. Skipping this task completely.")
        return []

    pearson_file = os.path.join(SAVE_BASE_PATH, f"Pearson_{subj_id}_{task_name}.h5")
    gca_file = os.path.join(SAVE_BASE_PATH, f"GCA_{subj_id}_{task_name}.h5")

    try:
        df = pd.read_parquet(file_path)
        df.dropna(axis=1, how='all', inplace=True)

        try:
            brain_info = {f"{net}_{reg}": net for net, reg in df.columns}
            df.columns = [f"{net}_{reg}" for net, reg in df.columns]
        except ValueError:
            brain_info = {col: col.split('_')[0] for col in df.columns}

        brain_regions = list(df.columns)
        raw_ts = df.values.T
        
        
        # Downsample MEG data (e.g., from 1000Hz to 200Hz) to reduce computation time for mTE
        ts_downsampled = decimate(raw_ts, q=5, axis=1) 
        
        # IDTxl requires multiple independent trials to infer probabilities correctly.
        # We simulate this by chopping the continuous resting-state data into 10 pseudo-trials.
        n_trials = 10
        max_len = (ts_downsampled.shape[1] // n_trials) * n_trials
        ts_downsampled = ts_downsampled[:, :max_len]
        samples_per_trial = ts_downsampled.shape[1] // n_trials 

        final_ts = ts_downsampled.reshape(len(brain_regions), n_trials, samples_per_trial)
        final_ts = final_ts.transpose(0, 2, 1)  # Final Shape required by IDTxl: (Processes, Samples, Replications)

        if os.path.exists(pearson_file):
            logging.info(f"[RESUME] Loading existing Pearson results...")
            with h5py.File(pearson_file, 'r') as f:
                corr_matrix = f['results/results/corr_matrix'][:]
                pearson_fdr_pval = f['results/results/pearson_fdr_pval'][:]
        else:
            corr_matrix, pearson_pval, pearson_fdr_pval = pearson_analysis(df, brain_regions)
            save_results(SAVE_BASE_PATH, f"Pearson_{subj_id}_{task_name}",
                         {'corr_matrix': corr_matrix, 'pval_matrix': pearson_pval,
                          'pearson_fdr_pval': pearson_fdr_pval},
                         brain_regions, brain_info, 'Pearson')

        if os.path.exists(gca_file):
            logging.info(f"[RESUME] Loading existing GCA results...")
            with h5py.File(gca_file, 'r') as f:
                gca_fdr_pval = f['results/results/gca_fdr_pval'][:]
        else:
            gca_pval, gca_fdr_pval = granger_analysis(df, brain_regions)
            save_results(SAVE_BASE_PATH, f"GCA_{subj_id}_{task_name}",
                         {'pval_matrix': gca_pval, 'gca_fdr_pval': gca_fdr_pval},
                         brain_regions, brain_info, 'GCA')

        sources_dict = select_mte_sources(corr_matrix, pearson_fdr_pval, gca_fdr_pval, brain_regions)
        local_summary[task_name] = list(sources_dict.keys()) if sources_dict else []

        mte_completed = False
        if os.path.exists(mte_file):
            try:
                with h5py.File(mte_file, 'r') as f:
                    if 'results/te_weighted_adjacency_matrix' in f:
                        mte_completed = True
            except: pass
            
        if mte_completed:
            logging.info(f"[SKIP] final mTE matrix exists, skipping!")
        else:
            if sources_dict:
                mte_results = mte_analysis(final_ts, brain_regions, sources_dict, brain_info, settings=SETTINGS, mte_file_path=mte_file)
                save_results(SAVE_BASE_PATH, f"mTE_{subj_id}_{task_name}", mte_results, brain_regions, brain_info, 'mTE')
            del mte_results

    except Exception as e:
        logging.error(f"Critical failure on {task_file}: {e}")
    finally:
        if 'df' in locals():
            del df
        gc.collect()

    logging.info(f"FINISH -> Subject {subj_id} | Task {task_name}")
    return local_summary


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="HCP MEG Brain Fingerprint - Parquet Parallel processing")
    parser.add_argument('--subj', type=str, required=True, help="'all' or comma-separated subject IDs")
    parser.add_argument('--parallel', type=int, default=1, help="Max parallel CPU workers")
    args = parser.parse_args()

    os.makedirs(SAVE_BASE_PATH, exist_ok=True)
    all_task_files = sorted([f for f in os.listdir(BASE_PATH) if f.endswith('.parquet')])

    if not all_task_files:
        print(f"No .parquet files found in {BASE_PATH}")
        sys.exit(1)

    subject_task_map = group_tasks_by_subject(all_task_files)
    all_subj_ids = sorted(subject_task_map.keys())
    selected_subj_ids = parse_subject_selection(args.subj, all_subj_ids)

    from multiprocessing import Pool

    tasks = []
    for subj_id in selected_subj_ids:
        for task_file in subject_task_map[subj_id]:
            task_name = task_file.replace('.parquet', '')
            tasks.append((subj_id, task_name, task_file))

    print(f"Total {len(tasks)} MEG tasks to run with {args.parallel} workers")

    all_summary = {}
    if args.parallel > 1 and len(tasks) > 1:
        with Pool(args.parallel) as p:
            for task_tuple, task_summary in zip(tasks, p.imap_unordered(run_one_task, tasks)):
                subj_id, task_name = task_tuple[0], task_tuple[1]
                if subj_id not in all_summary:
                    all_summary[subj_id] = {}
                all_summary[subj_id][task_name] = task_summary
    else:
        for t in tasks:
            subj_id, task_name = t[0], t[1]
            task_summary = run_one_task(t)
            if subj_id not in all_summary:
                all_summary[subj_id] = {}
            all_summary[subj_id][task_name] = task_summary

    if all_summary:
        save_mte_selection_summary(all_summary)
        print(f"Summary saved. Completed processing for {len(all_summary)} subjects.")